<p><font size="6" color='grey'> <b>
KI-Agenten. Planen. Handeln. Prüfen.
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Prompt Engineering für Agenten
</b></font> </br></p>

---

**Beitrag zum Leitprojekt:** Der System-Prompt legt fest, wie der Meeting- & Briefing-Agent seine Rolle versteht, wann er Tools einsetzt (**Planen und Handeln**) und wo er hart stoppen muss (**Prüfen** — siehe Hard Limits in Kapitel 6). Prompt Engineering ist damit keine reine Stil-Frage, sondern Teil der Steuerungslogik des Agentensystems.


**Praxisanker:** Mara Vogt braucht klare Prompts, damit der Agent ihre Briefing-Frage, die verfügbaren Dokumente und die Grenzen einer Antwort sauber auseinanderhält.


In [ ]:
#@title  🛠️ Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
# Globales Tracing aus: Nur der Referenzlauf in Kapitel 6 wird erfasst.
os.environ["LANGSMITH_TRACING"] = "false"
os.environ["LANGSMITH_PROJECT"] = "m03-prompt-engineering"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    mermaid,
    load_prompt,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# Modell-Konfiguration — Rollen als Konstanten
from genai_lib.model_config import BASELINE, ROUTER, JUDGE, PLANNER, WORKER, WORKER_PREMIUM, CODING, EMBEDDINGS

# 1 | Übersicht
---


Prompts sind die zentrale Steuerungsschicht zwischen Entwickler, Modell und Tools. Was im Prompt nicht klar geregelt ist, bleibt dem Modell überlassen: Rolle, Tool-Nutzung, Grenzen, Ausgabeformat und Stoppbedingungen.


In [ ]:
#@markdown   <p><font size="4" color='green'>  Prompt-Engineering-Workflow</font> </br></p>


# Prompt-Engineering-Workflow
diagram = """
%%{init: {'theme':'forest'}}%%
flowchart TB
    INPUT([Aufgabe]) --> SYSTEM[System-Prompt
Rolle & Kontext]
    INPUT --> FEW[Few-Shot
Beispiele]
    SYSTEM --> LLM[LLM]
    FEW --> LLM
    LLM --> TOOL{Tool
nötig?}
    TOOL -- ja --> CALL[Tool-Call
anfordern]
    CALL --> EXEC[Agent führt
Tool aus]
    EXEC -->|Ergebnis| LLM
    TOOL -- nein --> OUT([Antwort])
"""
mermaid(diagram, width=700)

Im Kurs wird das am Meeting-Briefing-Agent umgesetzt: derselbe fachliche Kern wird durch bessere Prompts zuverlässiger, kontrollierbarer und leichter prüfbar.

**Einordnung im Lernpfad:**

Die Prompt-Vorlage ist keine neue Chain, sondern die Steuerungsschicht, die festlegt, wann der Agent Tools nutzt, welche Grenzen gelten und wie er Ergebnisse prüfbar ausgibt. Die kurze Chain in Abschnitt 4 dient nur dazu, Prompt-Varianten isoliert zu vergleichen.


In [ ]:
# Imports für das Modul
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate

llm = init_chat_model(BASELINE)

**Prompt-Template: Rollen und Platzhalter**

Ein `ChatPromptTemplate` ist eine wiederverwendbare Vorlage für Nachrichten. Es kann zum Beispiel eine System-Nachricht mit Regeln und eine User-Nachricht mit der Frage enthalten. Platzhalter wie `{thema}` und `{zielgruppe}` werden erst beim Aufruf belegt.

Die nächste Zelle lädt eine solche Vorlage mit `load_prompt(..., mode="T")`. `prompt.invoke({...})` füllt die Platzhalter und liefert Nachrichten für das Modell. Dabei wird noch keine Modellantwort erzeugt.


In [ ]:
prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m03_research_template_prompt.md", mode="T")

messages = prompt.invoke({
    "thema": "Retrieval Augmented Generation",
    "zielgruppe": "Einsteiger"
}, config={"run_name": "m03_kap1_prompt_messages"})

for msg in messages.messages:
    print(f"{msg.type.upper()}: {msg.content}")

# 2 | System Prompts für Tool-Calling
---


Ein Agenten-Prompt verbindet die Prompt-Struktur mit konkreten Regeln für Tool-Calling. `ChatPromptTemplate` trennt die Rollen `system`, `user` und `ai` sowie Variablen klar voneinander. Das ist robuster als Prompt-Strings zusammenzukleben.

**Nachrichtentypen in LangChain:**

| Nachrichtentyp | Aufgabe |
|---|---|
| `SystemMessage` | Rolle, Regeln und Grenzen des Agenten |
| `HumanMessage` | Anfrage des Benutzers |
| `AIMessage` | Antwort oder Tool-Entscheidung des Modells |
| `ToolMessage` | Ergebnis eines Tool-Aufrufs |

Im einfachen Prompt-Beispiel werden diese Nachrichten über die Prompt-Vorlage erzeugt. Beim Agenten werden sie als Nachrichtenfolge sichtbar: `HumanMessage → AIMessage → ToolMessage → AIMessage`.

Prompts werden im Kurs als versionierte Markdown-Assets geladen:
- `load_prompt(..., mode="T")` liefert einen Chat-Prompt mit Rollen und Variablen.
- `load_prompt(..., mode="S")` liefert einen System-Prompt als Text für `create_agent()`.
- YAML-Frontmatter dokumentiert Name, Beschreibung und Variablen.

Für Agenten mit Tools sollte der `system`-Prompt explizit festlegen:
- wann ein Tool verwendet wird
- wann **kein** Tool verwendet wird
- welches Ausgabeformat erwartet wird


In [ ]:
#@markdown   <p><font size="4" color='green'>  Nachrichten-Typen Workflow</font> </br></p>

diagram = """
agent TD
    A["System Message"] --> D["LLM Verarbeitung"]
    B["User Message"] --> D
    D --> C["AI Message"]

    %% Tool Calling Loop
    C -- "tool_calls angefordert" --> T_EXEC["Tool Ausführung<br/>(Funktion / API)"]
    T_EXEC --> E["Tool Message"]
    E -- "Ergebnis / Rückgabe" --> D

    %% Erläuterungen
    A1["Definiert Verhalten<br/>Rolle & Persönlichkeit<br/>Einschränkungen"] -.-> A
    B1["Benutzer-Eingabe<br/>Frage oder Befehl<br/>Kontext"] -.-> B
    C1["Generierte Antwort<br/>Basierend auf System + User<br/>Optional: tool_calls"] -.-> C
    E1["Tool-Rückgabewert<br/>Verknüpft via tool_call_id<br/>Reines Ergebnis (z.B. JSON/Text)"] -.-> E

    %% Styling
    style A fill:#ffe1e1
    style B fill:#e1f5ff
    style C fill:#e1ffe1
    style D fill:#fff4e1
    style E fill:#f3e8ff
    style T_EXEC fill:#f0f0f0,stroke-dasharray: 5 5
"""
mermaid(diagram, width=700, height=450)

In [ ]:
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.tools import tool

In [ ]:
@tool
def anfrage_einordnen(text: str) -> str:
    """Extrahiert einfache Briefing-Signale aus einer Anfrage."""
    signale = [wort for wort in ["meeting", "briefing", "entscheidung", "risiko", "action item"] if wort in text.lower()]
    return ", ".join(signale) if signale else "Keine klaren Briefing-Signale gefunden."

@tool
def thema_einordnen(thema: str) -> str:
    """Schätzt ein, ob ein Thema zu den verfügbaren Meeting- und Projektdokumenten passt."""
    briefing_relevant = ["meeting", "briefing", "entscheidung", "risiko", "action item"]
    if any(wort in thema.lower() for wort in briefing_relevant):
        return "Thema passt zu den verfügbaren Meeting- und Projektdokumenten."
    return "Keine sichere Themenpassung zu den Meeting- und Projektdokumenten erkennbar."

In [ ]:
system_prompt = load_prompt("https://github.com/ralf-42/Agenten/blob/main/05_prompt/m03_research_system_prompt.md", mode="S")

agent = create_agent(
    model=WORKER,
    tools=[anfrage_einordnen, thema_einordnen],
    system_prompt=system_prompt,
)

result = agent.invoke({"messages": [HumanMessage(content="Prüfe, ob es zu RAG und Evaluation passende Meeting- und Projektdokumente gibt.")]}, config={"run_name": "m03_kap2_agent_result"})
mprint(result["messages"][-1].content)

# 3 | Few-Shot Examples
---


Few-Shot-Beispiele helfen Agenten, Tool-Auswahl und Antwortstruktur zu stabilisieren.
Verglichen werden ein Agent ohne Beispiele und ein Agent mit kurzen Beispielen für die Tool-Nutzung. Beide erhalten dieselbe Anfrage und dieselben Tools.


<p><font color='darkblue' size="4">
 <b>Viz</b>
</font></p>

- [Modellsteuerung](https://editor.p5js.org/ralf.bendig.rb/full/um423ggnD)
- [LLM-Parameter](https://editor.p5js.org/ralf.bendig.rb/full/LBc3t3yP4)


In [ ]:
# Zero-Shot- und Few-Shot-System-Prompts für denselben Agenten
base_agent_prompt = load_prompt(
    "https://github.com/ralf-42/Agenten/blob/main/05_prompt/m03_research_system_prompt.md",
    mode="S",
)

few_shot_agent_prompt = base_agent_prompt + """

Beispiele für die Tool-Auswahl:
- Anfrage: „Welche Research-Signale enthält diese Frage?“
  Aktion: Verwende `anfrage_einordnen`.
- Anfrage: „Passt dieses Thema zu den verfügbaren Meeting- und Projektdokumenten?“
  Aktion: Verwende `thema_einordnen`.

Nutze die Beispiele als Orientierung. Erfinde keine Tool-Ergebnisse.
"""

zero_shot_agent = create_agent(
    model=WORKER,
    tools=[anfrage_einordnen, thema_einordnen],
    system_prompt=base_agent_prompt,
)

few_shot_agent = create_agent(
    model=WORKER,
    tools=[anfrage_einordnen, thema_einordnen],
    system_prompt=few_shot_agent_prompt,
)

In [ ]:
frage = "Passt das Thema Prompt Caching zu den verfügbaren Meeting- und Projektdokumenten?"


def run(agent, frage, run_name):
    result = agent.invoke({"messages": [HumanMessage(content=frage)]}, config={"run_name": run_name})
    tools = [
        call["name"]
        for message in result["messages"]
        if hasattr(message, "tool_calls")
        for call in message.tool_calls
    ]
    print(f"Tool-Aufrufe = {tools or 'keine'}")
    print(result["messages"][-1].content.strip())


for name, current_agent in [("Zero-Shot", zero_shot_agent), ("Few-Shot", few_shot_agent)]:
    print(f"--- {name} ---")
    run(current_agent, frage, f"m03_kap3_{name.lower().replace('-', '_')}")
    print()

# 4 | Prompt-Varianten testen
---



Ein einfacher Vergleich hilft, bessere Prompts datenbasiert auszuwählen.
Unten testen wir mehrere System-Prompts auf dieselbe Anfrage.


In [ ]:
variants = {
    "knapp": "Antworte in maximal 2 Sätzen.",
    "didaktisch": "Erkläre schrittweise und nenne ein kurzes Beispiel aus dem Meeting-Briefing-Agent.",
    "kritisch": "Zeige Antwort, Unsicherheit und eine mögliche Fehlerquelle.",
}

variant_prompt = ChatPromptTemplate([
    ("system", "Rolle: Meeting-Briefing-Agent für den KI-Agenten-Kurs. {instruction}"),
    ("user", "{q}"),
])

In [ ]:
user_question = "Wann sollte ein Meeting-Briefing-Agent ein Retrieval-Tool nutzen?"

for _i, (name, instruction) in enumerate(variants.items(), 1):
    run_cfg = {
        "run_name": f"m03_kap4_variant_{name.lower()}",
        "tags": ["m03", "prompt-variants", "research", name.lower()]
    }
    answer = (variant_prompt | llm).invoke({"instruction": instruction, "q": user_question}, config={**run_cfg, "run_name": f"m03_kap4_answer_{_i}"})
    mprint("### Variante: " + name)
    mprint('---')
    mprint(answer.content.strip())

<p><font color='black' size="5">
Prompt-Pattern: Reasoning sichtbar machen, ohne interne Gedankenkette
</font></p>

Für Agenten-Prompts ist diese Formulierung robuster und sicherer:

```text
Arbeite mit sichtbaren Kontrollfeldern:
- plan: maximal 4 Schritte
- tool_choice: welches Tool und warum in einem Satz
- evidence_status: ausreichend | unsicher | nicht_im_korpus
- decision: antworten | nachrecherchieren | eskalieren

Gib keine ausführliche interne Gedankenkette aus.
```

Diese Felder werden später in LangGraph zu State-Feldern, Routing-Kriterien und Evaluationsmetriken.


# 5 | Strukturierte Agenten-Prompts: Production Pattern
---


Produktions-Prompts für Agenten folgen einem bewährten Muster mit 5 Patterns,
die Agenten zuverlässiger und kontrollierbarer machen.

> Diese Prompt-Regeln gelten auch für LangGraph-Workflows.



<p><font color='black' size="5">
Pattern 1: XML-Tags als Struktur-Trenner
</font></p>

LLMs folgen klar abgegrenzten Sektionen zuverlässiger als Fließtext.
Jede Sektion hat einen eigenen kognitiven Zweck.

```python
**Unstrukturiert: Modell muss Regeln aus Fließtext extrahieren**
system = "Rolle: Meeting-Briefing-Agent. Nutze Tools. Sei effizient. Stoppe wenn fertig."

**Strukturiert: klare Trennung von Rolle, Prozess und Grenzen**
system = """
<Task>Was der Agent tun soll</Task>
<Instructions>Wie er vorgehen soll</Instructions>
<Hard Limits>Wann er stoppen soll</Hard Limits>
"""
```

Fehlt die XML-Struktur, übersieht das Modell häufiger Regeln, die weiter unten im Prompt stehen - besonders bei langen System-Prompts mit mehr als 500 Tokens.




<p><font color='black' size="5">
Pattern 2: Explizite Tool-Steuerung
</font></p>

Ohne Anweisung nutzen Agenten Tools opportunistisch. Explizite Reflexions-Aufforderung erzwingt Bewertung nach jedem Schritt.

```python
"KRITISCH: Nach jedem Tool-Ergebnis bewerten: Reicht der Kontext für eine belegbare Antwort?"
```




<p><font color='black' size="5">
Pattern 3: Quantitative Limits statt qualitativer Anweisungen
</font></p>

```python
**Qualitativ: nicht handlungsleitend genug**
"Sparsam mit Tool-Calls umgehen."

**Quantitativ: konkret und messbar**
"Tool-Budget: maximal 3 Retrieval-Aufrufe. Nach 3 Aufrufen immer stoppen."
```



<p><font color='black' size="5">
Pattern 4: Explizite Stop-Bedingungen
</font></p>

Agenten neigen zu Over-Research. Ohne Stop-Kriterien führen sie unnötige weitere Tool-Aufrufe aus.

```python
"""Sofort stoppen wenn:
- Die Frage vollständig mit verfügbarem Kontext beantwortet werden kann
- Die letzte Suche keine neuen Informationen brachte
- 3 relevante Quellen gefunden wurden
"""
```




<p><font color='black' size="5">
Pattern 5: Strategie vorgeben (breit -> eng)
</font></p>

Für Recherche-Agenten ist eine explizite Suchstrategie wichtig:
+ zuerst grob einordnen,
+ dann gezielt vertiefen,
+ danach Antwort synthetisieren.


In [ ]:
# Strukturierter System-Prompt mit prüfbaren Feldern
structured_system_prompt = """

Rolle: Meeting-Briefing-Agent für den KI-Agenten-Kurs.

<Task>
Beantworte Fachfragen zu RAG, Retrieval, Evaluation und Agenten knapp und nachvollziehbar.
</Task>

<Instructions>
1. Frage einordnen: Begriff, Vergleich, Anwendung oder außerhalb des Korpus?
2. Tool nur nutzen, wenn Briefing-Signale oder Themenpassung zu Dokumenten geprüft werden müssen.
3. Nach jedem Tool-Ergebnis prüfen: Reicht der Kontext für eine belegbare Antwort?
4. Unsicherheit ausdrücklich nennen, wenn die Themenpassung zu Dokumenten unklar ist.
</Instructions>

<Hard Limits>
Tool-Budget: maximal 2 Tool-Aufrufe.
Maximale Antwortlänge: 120 Wörter.
Sofort antworten, wenn Begriff und Korpusbezug klar sind.
</Hard Limits>"""

In [ ]:
@tool
def quellenhinweis_erstellen_demo(thema: str) -> str:
    """Gibt einen Demo-Hinweis auf mögliche Quellenbindung zurück."""
    return f"Für '{thema}' sollte die Antwort mit Fundstelle, Dokumenttitel, Abschnitt oder Dokument-ID belegt werden."

In [ ]:
agent = create_agent(
    model=WORKER,
    tools=[quellenhinweis_erstellen_demo],
    system_prompt=structured_system_prompt,
)

result = agent.invoke({"messages": [HumanMessage(content="Wie sollte ein Meeting-Briefing-Agent Antworten zu RAG belegen?")]}, config={"run_name": "m03_kap6_agent_result"})
mprint(result["messages"][-1].content)

# 6 | LangSmith: Prompt-Inspektion
---



LangSmith macht Agenten-Prompt-Experimente nachvollziehbar: Welcher System-Prompt führte zu welcher Tool-Entscheidung und Antwort?

**Empfehlung im Kursbetrieb:**
- `run_name` pro Experiment setzen
- `tags` für Modul/Kapitel verwenden
- bei Vergleichen immer denselben Input nutzen
- nur den Referenzlauf explizit mit LangSmith tracen, damit keine unnötigen Traces entstehen

**Hinweis (EU):** Endpoint `https://eu.api.smith.langchain.com` – bereits in der Setup-Cell konfiguriert.

In [ ]:
# 1. Tracing-Konfiguration vorab festlegen
run_cfg = {
    "run_name": "m03_kap7_agentpromptinspection",
    "tags":     ["m03", "prompt-engineering", "agent", "research"]
}

In [ ]:
# 2. Nur den Agenten-Referenzlauf in einem expliziten Tracing-Kontext ausführen
from langsmith import tracing_context

with tracing_context(enabled=True):
    traced_response = agent.invoke({
        "messages": [
            HumanMessage(
                content="Prüfe, wie ein Meeting-Briefing-Agent mit einer Frage außerhalb des Korpus umgehen soll."
            )
        ]
    }, config={**run_cfg, "run_name": "m03_kap7_traced_response"})

mprint("### Agenten-Trace erstellt")
mprint("---")
mprint(traced_response["messages"][-1].content)

In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("m03-prompt-engineering", limit=3, show_steps=True)

# A | Aufgaben
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabenstellungen unten bieten Anregungen; alternative Herausforderungen sind möglich.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs darf und soll generative KI auch als Unterstützung beim Lernen und Entwickeln genutzt werden. Geeignet ist sie zum Beispiel, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.


**Grundlagen**
- Einen klaren System-Prompt für einen einfachen Meeting-Briefing-Agent schreiben.
- Mit 1-2 Beispielanfragen testen.


In [ ]:
# Grundlagen: System-Prompt + Agent
from langchain_core.tools import tool

@tool
def search_briefing_context(thema: str) -> str:
    """Gibt Demo-Informationen zum Meeting-Briefing-Kontext zurück."""
    pass

@tool
def thema_einordnen_exercise(thema: str) -> str:
    """Schätzt in der Demo die Themenpassung zu Meeting- und Projektdokumenten."""
    pass

**Aufbau**
1. Einen Agenten mit 2-3 Briefing-Tools erstellen, zum Beispiel Dokumentenpassung, Signalextraktion und Quellenhinweis.
2. Zwei unterschiedliche System-Prompts schreiben: einmal knapp, einmal vorsichtig mit Unsicherheitsmarkierung.
3. 3 Testfragen definieren und beide Varianten vergleichen.
4. Kurz dokumentieren, welche Prompt-Version für den Meeting-Briefing-Agent besser passt und warum.


In [ ]:
# Aufbau: Zwei Prompt-Varianten vergleichen
meine_varianten = {
    "knapp": "...",
    "vorsichtig": "...",
}

pass

**Vertiefung**
- Ein Few-Shot-Beispiel in einer der beiden Prompt-Varianten ergänzen.
- Eine klare Regel formulieren, wann der Agent Tools vermeiden soll.


In [ ]:
# Vertiefung: Few-Shot + Tool-Vermeidungsregel
mein_few_shot_prompt = """Beispiel 1:
Frage: Was bedeutet RAG?
Antwort: definition

Beispiel 2:
Frage: Wie unterscheidet sich RAG von Fine-Tuning?
Antwort: vergleich
"""

# ...

# B | Dokumente zum Weiterlesen
---



Ergänzende Artikel aus der Kurs-Dokumentation:

- [Prompt Engineering](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/prompt-engineering.html)
- [Einsteiger-Prompts](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/einsteiger-prompts.html)
- [Tool Use & Function Calling](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/tool-use-function-calling.html)
- [Agenten-Architekturen](https://ralf-42.github.io/Agenten/04-agenten-implementierung/entwurf/agent-architekturen.html)
- [LangChain Best Practices](https://ralf-42.github.io/Agenten/05-frameworks/langchain-best-practices.html)